In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import io

class ImageNetParquetDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_bytes = row["image"]["bytes"]
        label = row["label"]

        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label


In [ ]:
import glob
import pandas as pd

files = sorted(glob.glob("/home/shared/data/imagenet/validation-*.parquet"))
print(files)  # sanity check

df = pd.read_parquet(files)

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from abc import ABC, abstractmethod
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import torch
import pickle


def pairwise_cosine_similarity(x1, x2):
    """
    Compute pairwise cosine similarity between two sets of vectors.
    
    Args:
        x1: First set of vectors [N, D]
        x2: Second set of vectors [M, D]
        
        Similarity matrix [N, M] where entry (i,j) is cosine similarity between x1[i] and x2[j]
    """
    # Normalize vectors to unit length
    x1_norm = F.normalize(x1, p=2, dim=1)
    x2_norm = F.normalize(x2, p=2, dim=1)
    
    # Compute cosine similarity via matrix multiplication
    similarity_matrix = torch.mm(x1_norm, x2_norm.t())
    
    return similarity_matrix

def compute_rewards(msg_repr, img_repr, contrastive_loss_temperature, idx=None, device='cuda'):
    if isinstance(idx, torch.Tensor):
        similarities_messages_to_objects = pairwise_cosine_similarity(msg_repr, img_repr) / contrastive_loss_temperature
        rewards = -1 * F.cross_entropy(similarities_messages_to_objects, torch.arange(0, img_repr.shape[0]).to(device), reduction='none').detach()
    else:
        text_i = msg_repr[idx].unsqueeze(0)
        similarities = pairwise_cosine_similarity(
            text_i, 
            img_repr
        ) / contrastive_loss_temperature
        target = torch.tensor([idx], device=device)
        rewards = -1 * F.cross_entropy(similarities, target, reduction='none').detach()
    
    return rewards

In [ ]:
compute_rewards(torch.randn(32, 1024).to('cuda'), torch.randn(32, 1024).to('cuda'), 1, idx=torch.tensor([1, 2])).shape

In [ ]:
import numpy as np

SEED = 42
TEST_SAMPLES_PER_CLASS = 32

# Shuffle within each class, then split
test_df = (
    df.groupby("label", group_keys=False)
      .apply(lambda x: x.sample(n=TEST_SAMPLES_PER_CLASS, random_state=SEED))
)

train_df = df.drop(test_df.index)

print("Train size:", len(train_df))
print("Test size:", len(test_df))


In [ ]:
test_df["label"].value_counts()


In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import io

class ImageNetParquetDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_bytes = row["image"]["bytes"]
        label = row["label"]

        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label


In [ ]:
import torch
from torchvision import models

weights = models.ResNet50_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
preprocess

In [ ]:
dataset = ImageNetParquetDataset(
    df,
    transform=preprocess
)

In [ ]:
import torch
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"

weights = models.ResNet50_Weights.IMAGENET1K_V1
model = models.resnet50(weights=weights)

# Remove classification head
model.fc = torch.nn.Identity()
model = model.to(device)
model.eval()


In [ ]:
from PIL import Image
import io
import matplotlib.pyplot as plt

# Get the image bytes
img_bytes = df.loc[10, 'image']['bytes']

# Convert bytes → PIL Image
img = Image.open(io.BytesIO(img_bytes))

# Plot
plt.imshow(img)
plt.axis("off")
plt.show()


---

In [ ]:
train_dataset = PreSavedBatchDataset(torch.load("/home/shared/data/shape_unique_single_attribute_old/train.pt"))
val_dataset = PreSavedBatchDataset(torch.load("/home/shared/data/shape_unique_single_attribute_old/validation.pt"))
test_dataset = PreSavedBatchDataset(torch.load("/home/shared/data/shape_unique_single_attribute_old/test.pt"))

In [ ]:
def plot_images(imgs):
    n = len(imgs)
    cols = 5
    rows = int(np.ceil(n / cols))
    
    plt.figure(figsize=(cols * 3, rows * 3))
    
    for i, img in enumerate(imgs):
        plt.subplot(rows, cols, i + 1)
        
        # Handle grayscale vs RGB
        if img.ndim == 2:
            plt.imshow(img, cmap="gray")
        else:
            plt.imshow(img)
        
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()

In [ ]:
len(val_dataset)

In [ ]:
plot_images(train_dataset[random.randint(0, 70)][0])

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import random    

class SingleDigitPadded(Dataset):
    def __init__(self, mnist_dataset):
        self.mnist = mnist_dataset

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        img, label = self.mnist[idx]

        # Decide randomly whether the digit is on the left or right
        if random.random() < 0.5:
            # digit on left
            empty = torch.zeros_like(img)
            image = torch.cat((img, empty), dim=2)  # 28x56
            position = "left"
        else:
            # digit on right
            empty = torch.zeros_like(img)
            image = torch.cat((empty, img), dim=2)
            position = "right"

        return image, label, position
    

class TwoDigitOpposite(Dataset):
    def __init__(self, padded_dataset):
        """
        padded_dataset: your SingleDigitPadded dataset
        """
        self.images = padded_dataset["images"]
        self.positions = padded_dataset["positions"]
        self.labels = padded_dataset["labels"]

        # Precompute indices by position for faster sampling
        self.left_indices = [i for i in range(len(self.images)) if self.positions[i] == "left"]
        self.right_indices = [i for i in range(len(self.images)) if self.positions[i] == "right"]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # First sample (fixed)
        img1, label1, pos1 = self.images[idx], self.labels[idx], self.positions[idx]

        # Determine opposite position
        opposite_pos = "right" if pos1 == "left" else "left"
        candidate_indices = self.right_indices if opposite_pos == "right" else self.left_indices

        # Randomly select second sample with opposite position
        random.seed(42)
        idx2 = random.choice(candidate_indices)
        img2, label2 = self.images[idx2], self.labels[idx2]

        # Combine images by adding (keep same shape)
        two_digit_img = img1 + img2
        two_digit_label = (label1, label2) if pos1 == "left" else (label2, label1)

        return two_digit_img, two_digit_label

    
def create_mnist1():
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])

    train_dataset = datasets.MNIST(
        root="/home/shared/data",
        train=True,
        download=True,
        transform=transform
    )

    test_dataset = datasets.MNIST(
        root="/home/shared/data",
        train=False,
        download=True,
        transform=transform
    )


    singleDigit_train = SingleDigitPadded(train_dataset)
    singleDigit_test = SingleDigitPadded(test_dataset)

    train_loader = DataLoader(singleDigit_train, batch_size=len(singleDigit_train))  
    test_loader = DataLoader(singleDigit_test, batch_size=len(singleDigit_test)) 

    for images, labels, positions in train_loader:
        torch.save({
            'images': images,        # Tensor [N, 1, 28, 56]
            'labels': labels,
            'positions': positions  
        }, '/home/shared/data/MNIST2/train.pt')
        break  # only one batch needed

    for images, labels, positions in test_loader:
        torch.save({
            'images': images,        # Tensor [N, 1, 28, 56]
            'labels': labels,
            'positions': positions
        }, '/home/shared/data/MNIST2/test.pt')
        break  # only one batch needed


def create_mnist2():
    singleDigit = torch.load('/home/shared/data/MNIST1/test.pt')
    
    two_digit_dataset = TwoDigitOpposite(singleDigit)

    # Save the dataset
    loader = DataLoader(two_digit_dataset, batch_size=len(two_digit_dataset))

    for images, labels in loader:
        torch.save({
            'images': images,        # Tensor [N, 1, 28, 56]
            'labels': labels[0] * 10 + labels[1],
        }, '/home/shared/data/MNIST2/test.pt')
        break

In [2]:
create_mnist2()

In [ ]:
import torch
singleDigit = torch.load('/home/shared/data/MNIST2/test.pt')

In [ ]:
import matplotlib.pyplot as plt
import random
i = random.randint(0, 2000)
plt.imshow(singleDigit['images'][i].permute(1, 2, 0), cmap='gray')
plt.title(singleDigit['labels'][i])
plt.axis('off')
plt.show()

# Analysis

In [ ]:
from itertools import combinations, chain, product
import random
from collections import defaultdict
from torch.utils.data import random_split
from functools import reduce
import operator


def SolveMinSym(target_image, all_images):
    """
    Minimum number of positions required to uniquely
    identify the target image.
    """
    distracting_images = [
        img for img in all_images if img != target_image
    ]

    for combination in attribute_combinations(target_image):
        if is_unique_combination(combination, target_image, distracting_images):
            return len(combination)

    return None


def attribute_combinations(image):
    """
    Generate combinations of attribute INDICES
    (not values).
    """
    indices = list(range(len(image)))

    return chain.from_iterable(
        combinations(indices, r)
        for r in range(1, len(indices) + 1)
    )


def is_unique_combination(combination, target_image, distracting_images):
    """
    Check if selected positions uniquely identify target.
    """
    for image in distracting_images:
        match = all(
            image[idx] == target_image[idx]
            for idx in combination
        )

        if match:
            return False

    return True


def min_m_controlled_sampling(dataset, number_of_samples, target_min_symbol, batch_size):

    result = []
    while (len(result) * batch_size) < number_of_samples:

        batch = random.sample(dataset, batch_size)
        target = random.sample(batch, 1)[0]
        
        batch_indices = [image[1] for image in batch]
        batch_values = [image[0] for image in batch]
        target_value, target_index = target
        
        min_symbol = SolveMinSym(target_value, batch_values)
        
        if min_symbol == target_min_symbol:
            result.append({target_value: batch_values})

    return reduce(operator.or_, result)

In [ ]:

from torch.utils.data import Dataset
import torch
import torch.nn.functional as F

        
class ObjectsDataset(Dataset):
    def __init__(self, num_attributes=4, num_values=10, indices=None, min_symbol=2, batch_size=32):
        self.num_attributes = num_attributes
        self.num_values = num_values
        self.data = dict(enumerate(product(range(num_values), repeat=num_attributes)))
        
        if indices is None:
            indices = list(self.data.keys())
        
        self.data = {i: self.data[k] for i, k in enumerate(indices)}
        
        self.candidates = min_m_controlled_sampling(
            [(self.data[k], k) for k in self.data.keys()],
            number_of_samples=len(self.data),
            target_min_symbol=min_symbol,
            batch_size=batch_size
        )

        self.samples = []
        for target, cand in self.candidates.items():
            self.samples.append((cand, cand.index(target)))
            
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        cands, target = self.samples[idx]
        one_hots = torch.stack([self.__to_one_hot(torch.tensor(cand)) for cand in cands])
        return one_hots, target
    
    def get_candidates(self):
        return self.candidates
    
    def __to_one_hot(self, x):
        one_hot = F.one_hot(x, num_classes=self.num_values)
        return one_hot.flatten().float()  
    

train, val, test = random_split(range(10000), [0.8, 0.1, 0.1])

train_dataset = ObjectsDataset(indices=train.indices)
val_dataset = ObjectsDataset(indices=val.indices)
test_dataset = ObjectsDataset(indices=test.indices)


In [ ]:
train_dataset[0][0].shape

In [ ]:
from torch.utils.data import Dataset
import torchvision.transforms as transforms
import numpy as np
import os
import torch
import shapeworld
from PIL import Image
import math
import os
import pickle
from torch.utils.data import Dataset
import torch
import torch.nn.functional as F
from itertools import product
from solve_min_sym import min_m_controlled_sampling

In [ ]:
class ShapeWorld(Dataset):
    def __init__(
        self, 
        n=128, 
        mode='train', 
        dtype='agreement', 
        name='existential', 
        collision_tolerance=0.0, 
        config='/home/shared/ShapeWorld/configs/agreement/existential/oneshape_test.json',
        **kwargs
    ):
        dataset = shapeworld.Dataset.create(dtype=dtype, name=name, collision_tolerance=collision_tolerance, config=config, **kwargs)
        generated = dataset.generate(n=n, mode=mode, include_model=True)
        self.imgs = generated['world']
    
    def __len__(self):
        return self.imgs.shape[0]

    def __getitem__(self, idx):
        return torch.tensor(self.imgs[idx]), 2

In [ ]:
from torch.utils.data import DataLoader
import torch

train_dataset = ShapeWorld()

loader = DataLoader(train_dataset, batch_size=32)

In [ ]:
for target, labels in loader:
    print(target.shape)
    if (labels == -1).all():
        print("None")
    # print(target[0])
    # print(labels[0])
    break

In [ ]:
import torch

vector = torch.tensor([1, 2, 3])



In [ ]:
from torch.utils.data import DataLoader, random_split
train, val, test = random_split(range(10), [0.8, 0.1, 0.1])

In [ ]:
list(test)

In [ ]:

              # shape: [40]

x = torch.tensor([3, 0, 9, 2])
encoded = encode_one_hit(x)

print(encoded.shape)  # torch.Size([40])


In [ ]:
for target, batch in test_dataset:
    print("Target:", target)
    print("Batch:", batch)
    # break

In [ ]:
3978 in test.indices

In [ ]:
test_dataset.min_m_controlled_sampling(min_symbol=3, batch_size=100)

In [ ]:
for target, batch in result.items():
    print(SolveMinSym(target, batch))

In [ ]:
len(set(result[i][1] for i in range(len(result))))

In [ ]:
(3, 5, 6, 5) == (3, 5, 6, 5)

In [ ]:
for target in result[3][0]:
    print(SolveMinSym(target, result[3][0]))

In [ ]:
for i in attribute_combinations([1, 2, 3]):
    print(i)

In [ ]:

import torch
print(len(vectors))  # 10000
print(vectors[:10])  # first 10 vectors


---

In [9]:
! pip install git+https://github.com/facebookresearch/EGG.git

  Cloning https://github.com/facebookresearch/EGG.git to /tmp/pip-req-build-wnkioznd
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/EGG.git /tmp/pip-req-build-wnkioznd
  Resolved https://github.com/facebookresearch/EGG.git to commit 54eb11c6b4ba3d3171811378f7ad7a117fa93a97
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 312.7 kB/s  0:00:08 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 288.4 kB/s  0:01:33m0:00:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 257.4 kB/s  0:00:08 eta 0:00:01
  Created wheel for EGG: filename=egg-0.1.0-py3-none-any.whl size=219793 sha256=d6e30134f8ac0354ec5aab76ae22121086d7cf603e5224c1f6993ae5df0882fe
  Stored in directory: /tmp/pip-ephem-wheel-cache-p1pnm7nx/wheels/ed/6a/b0/8b528e5fc900d600d48ea342ce00275f8ed55e6a396d9e1c87
Successfully

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import egg.core as core
from egg.core import GumbelSoftmaxWrapper

from adamw_schedule_free import AdamWScheduleFree


# =====================
# Dataset
# =====================

class ToyReferentialDataset(Dataset):
    """
    Random vectors as objects.
    Each sample contains:
        - target
        - distractors
        - label (index of target in candidates)
    """

    def __init__(self, n_objects=1000, object_dim=10, n_distractors=3):
        super().__init__()
        self.object_dim = object_dim
        self.n_distractors = n_distractors
        self.n_candidates = n_distractors + 1

        self.objects = torch.randn(n_objects, object_dim)

    def __len__(self):
        return len(self.objects)

    def __getitem__(self, idx):
        target = self.objects[idx]

        distractor_indices = torch.randint(
            low=0, high=len(self.objects), size=(self.n_distractors,)
        )
        distractors = self.objects[distractor_indices]

        candidates = torch.cat([target.unsqueeze(0), distractors], dim=0)

        label = torch.tensor(0)  # target always at index 0

        return target, candidates, label


# =====================
# Sender
# =====================

class Sender(nn.Module):
    def __init__(self, input_dim, hidden_dim, vocab_size):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
        )

    def forward(self, x):
        logits = self.fc(x)
        return logits


# =====================
# Receiver
# =====================

class Receiver(nn.Module):
    def __init__(self, vocab_size, object_dim, hidden_dim):
        super().__init__()

        self.msg_embedding = nn.Linear(vocab_size, hidden_dim)

        self.object_encoder = nn.Sequential(
            nn.Linear(object_dim, hidden_dim),
            nn.ReLU()
        )

    def forward(self, message, candidates):
        """
        message: (batch, vocab_size) one-hot (Gumbel)
        candidates: (batch, n_candidates, object_dim)
        """

        msg_vec = self.msg_embedding(message)

        batch_size, n_candidates, _ = candidates.size()

        candidates = candidates.view(-1, candidates.size(-1))
        obj_vec = self.object_encoder(candidates)
        obj_vec = obj_vec.view(batch_size, n_candidates, -1)

        scores = torch.bmm(obj_vec, msg_vec.unsqueeze(-1)).squeeze(-1)

        return scores


# =====================
# Game Wrapper
# =====================

class ReferentialGame(nn.Module):
    def __init__(self, sender, receiver):
        super().__init__()
        self.sender = sender
        self.receiver = receiver

    def forward(self, sender_input, labels, receiver_input):
        message_logits = self.sender(sender_input)

        scores = self.receiver(message_logits, receiver_input)

        loss = F.cross_entropy(scores, labels)

        acc = (scores.argmax(dim=1) == labels).float().mean()

        return loss, {"acc": acc}


# =====================
# Training Setup
# =====================

def main():
    opts = core.init()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    object_dim = 10
    hidden_dim = 128
    vocab_size = 10
    n_distractors = 3

    dataset = ToyReferentialDataset(
        n_objects=2000,
        object_dim=object_dim,
        n_distractors=n_distractors
    )

    loader = DataLoader(dataset, batch_size=64, shuffle=True)

    sender = Sender(object_dim, hidden_dim, vocab_size)
    sender = GumbelSoftmaxWrapper(sender)

    receiver = Receiver(vocab_size, object_dim, hidden_dim)

    game = ReferentialGame(sender, receiver)

    optimizer = AdamWScheduleFree([
        {"params": sender.parameters(), "lr": 1e-4},
        {"params": receiver.parameters(), "lr": 1e-3},
    ])

    trainer = core.Trainer(
        game=game,
        optimizer=optimizer,
        train_data=loader,
        device=device,
    )

    trainer.train(n_epochs=20)


if __name__ == "__main__":
    main()


In [6]:
import torch

vector = torch.zeros(32)
vector[2] = 0
(vector == 0).all()

tensor(True)

In [ ]:
F.cross_entropy

In [43]:
from itertools import combinations, chain, product
import random
from collections import defaultdict
from torch.utils.data import random_split
from functools import reduce
import operator


def SolveMinSym(target_image, all_images):
    """
    Minimum number of positions required to uniquely
    identify the target image.
    """
    distracting_images = [
        img for img in all_images if img != target_image
    ]

    for combination in attribute_combinations(target_image):
        if is_unique_combination(combination, target_image, distracting_images):
            return len(combination)

    return None


def attribute_combinations(image):
    """
    Generate combinations of attribute INDICES
    (not values).
    """
    indices = list(range(len(image)))

    return chain.from_iterable(
        combinations(indices, r)
        for r in range(1, len(indices) + 1)
    )


def is_unique_combination(combination, target_image, distracting_images):
    """
    Check if selected positions uniquely identify target.
    """
    for image in distracting_images:
        match = all(
            image[idx] == target_image[idx]
            for idx in combination
        )

        if match:
            return False

    return True


def min_m_controlled_sampling(dataset, number_of_samples, target_min_symbol, batch_size):

    result = []
    while (len(result) * batch_size) < number_of_samples:

        batch = random.sample(dataset, batch_size)
        target = random.sample(batch, 1)[0]
        
        batch_indices = [image[1] for image in batch]
        batch_values = [image[0] for image in batch]
        target_value, target_index = target
        
        min_symbol = SolveMinSym(target_value, batch_values)
        
        if min_symbol == target_min_symbol:
            result.append({target_value: batch_values})

    return reduce(operator.or_, result)

In [122]:
from torch.utils.data import Dataset
import torchvision.transforms as transforms
import numpy as np
import os
import torch
from PIL import Image
import math
import os
import pickle
from torch.utils.data import Dataset
import torch
import torch.nn.functional as F
from itertools import product

class ObjectsDataset(Dataset):
    def __init__(self, num_attributes=4, num_values=10, indices=None, min_symbol=2, batch_size=32, number_of_samples=None):
        self.num_attributes = num_attributes
        self.num_values = num_values
        self.data = dict(enumerate(product(range(num_values), repeat=num_attributes)))
        
        if indices is None:
            indices = list(self.data.keys())
        
        self.data = {i: self.data[k] for i, k in enumerate(indices)}
        
        self.candidates = min_m_controlled_sampling(
            [(self.data[k], k) for k in self.data.keys()],
            number_of_samples=number_of_samples,
            target_min_symbol=min_symbol,
            batch_size=batch_size
        )

        self.samples = []
        for target, cand in self.candidates.items():
            self.samples.append((cand, cand.index(target)))
            
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        cands, target_idx = self.samples[idx]
        return np.array(cands), target_idx
        # return one_hots, torch.tensor(-1).to(device='cuda')
    

In [130]:
def get_items(dataset):
    batchs = []
    targets = []
    for batch, target in dataset:
        batchs.append(batch)
        targets.append(target)
    data = np.stack(batchs)
    labels = np.array(targets)
    return data, labels


num_values = 20
train, val, test = random_split(range(num_values**4), [0.8, 0.1, 0.1])

train_dataset = ObjectsDataset(indices=train.indices, num_values=num_values, min_symbol=2, batch_size=32, number_of_samples=80000)
val_dataset = ObjectsDataset(indices=val.indices, num_values=num_values, min_symbol=2, batch_size=32, number_of_samples=10000)
test_dataset = ObjectsDataset(indices=test.indices, num_values=num_values, min_symbol=3, batch_size=100, number_of_samples=10000)

In [128]:
import numpy as np

# Stack everything
data = np.stack(batchs)          # shape: [N, n_distractors+1, n_features]
labels = np.array(targets)       # shape: [N]

print("Data shape:", data.shape)
print("Labels shape:", labels.shape)

# -------- Split --------
train, train_labels = get_items(train_dataset)
valid, valid_labels = get_items(val_dataset)
test, test_labels = get_items(test_dataset)

# -------- Save --------
np.savez(
    "objects_dataset3.npz",
    train=train,
    train_labels=train_labels,
    valid=valid,
    valid_labels=valid_labels,
    test=test,
    test_labels=test_labels,
)

print("Dataset saved as objects_dataset.npz")


Data shape: (9999, 32, 4)
Labels shape: (9999,)
Dataset saved as objects_dataset.npz
